Librerías y configuración base

In [ ]:
# === BLOQUE 0 · LIBRERÍAS Y CONFIG ===
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


1) Columnas canónicas y archivo de entrada

In [ ]:
# === BLOQUE 1 · COLUMNAS Y ARCHIVO ===
COL_POZO   = "name_"
COL_FECHA  = "date"
COL_QTOT   = "prueba_pozooil_24__prueba_pozowater_24"
COL_QGAS   = "prueba_de_producción_gas_a_24_horas_mcfd"
COL_HZ     = "frecuencia_bomba_hz"
COL_PINT   = "presion_de_intake_psi"
COL_BOMBA  = "tipo_de_bomba"
COL_REG    = "regimen_id"     # se construye si no existe

RUTA = r"C:\Users\Vìctor\OneDrive\Desktop\ESP_Project"
FILE = "df_con_eventos_hibridos.xlsx"   # tu dataset actual

df = pd.read_excel(Path(RUTA)/FILE)
print(">> Columnas originales:")
print(df.columns.tolist())


Saneo mínimo (sin cambiar nombres de dataset)

In [ ]:
# === BLOQUE 2 · SANEO MÍNIMO ===
# Limpia espacios invisibles en NOMBRES de columnas (no cambia tus nombres)
df.columns = df.columns.str.strip()

# Asegura tipo fecha si existe y ordena por pozo-fecha
if COL_FECHA in df.columns:
    df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors="coerce")
    df = df.sort_values([COL_POZO, COL_FECHA], na_position="last").reset_index(drop=True)

# Evidencias
print("\n>> Primeras filas (solo claves):")
print(df[[c for c in [COL_POZO, COL_FECHA, COL_BOMBA] if c in df.columns]].head(8))


Reconstrucción robusta de regimen_id

In [ ]:
# === BLOQUE 3 · RECONSTRUIR regimen_id (si falta) ===

def _normalizar_bomba(x: object) -> object:
    """
    Normaliza la nomenclatura de bomba:
    - unifica separadores ('|', '/', '\\', ',', ';') a '|'
    - toma el primer token como 'modelo nominal'
    - preserva NaN si no hay dato
    """
    if pd.isna(x): 
        return np.nan
    s = str(x).strip()
    for sep in ["|", "/", "\\", ",", ";"]:
        s = s.replace(sep, "|")
    return s.split("|")[0].strip() if s else np.nan

if COL_REG not in df.columns:
    print("\n>> 'regimen_id' no existe; se construirá a partir de pozo + tipo_de_bomba.")
    if COL_BOMBA in df.columns:
        df["bomba_norm"] = df[COL_BOMBA].apply(_normalizar_bomba)
    else:
        df["bomba_norm"] = np.nan

    df[COL_REG] = np.where(
        df["bomba_norm"].notna(),
        df[COL_POZO] + "||" + df["bomba_norm"],
        df[COL_POZO]
    )
else:
    # Asegura limpieza si ya existía
    df[COL_REG] = df[COL_REG].astype(str).str.strip()
    if "bomba_norm" not in df.columns and COL_BOMBA in df.columns:
        df["bomba_norm"] = df[COL_BOMBA].apply(_normalizar_bomba)

# Colapsa regímenes con muy pocos datos al nivel pozo
MIN_REG = 15  # ajustable 10–20
sizes = df.groupby(COL_REG).size()
pocos = set(sizes[sizes < MIN_REG].index)
if pocos:
    mask_pocos = df[COL_REG].isin(pocos)
    df.loc[mask_pocos, COL_REG] = df.loc[mask_pocos, COL_POZO]
    print(f">> {len(pocos)} regímenes con <{MIN_REG} filas colapsados a nivel pozo.")

print("\n>> Chequeo de claves del groupby:")
for col in [COL_POZO, COL_REG]:
    if col not in df.columns:
        raise KeyError(f"Falta la columna requerida: {col}")
print("OK: existen", COL_POZO, "y", COL_REG)
print(df[[COL_POZO, "bomba_norm", COL_REG]].head(10))


4) Definición de percentiles del “sobre operativo”

In [ ]:
# === BLOQUE 4 · SOBRE OPERATIVO (percentiles por pozo × régimen) ===

Q_LOW      = 0.10   # percentil inferior de Qtot
Q_HIGH     = 0.90   # percentil superior de Qtot
Q_GAS_HI   = 0.90   # percentil alto de gas
Q_HZ_HI    = 0.90   # percentil alto de Hz (apoyo a gating)
Q_PINT_LO  = 0.10   # percentil bajo de P_intake (apoyo a gating)

def _q(s: pd.Series, q: float):
    s = s.dropna()
    return np.nan if s.empty else np.nanpercentile(s, q*100)

grp_reg = df.groupby([COL_POZO, COL_REG])
sizes_reg = grp_reg.size().sort_values(ascending=False).head(10)
print("\n>> Top 10 regímenes por tamaño:")
print(sizes_reg)

q_reg = grp_reg.agg(
    qtot_lo = (COL_QTOT,  lambda s: _q(s, Q_LOW)),
    qtot_hi = (COL_QTOT,  lambda s: _q(s, Q_HIGH)),
    gas_hi  = (COL_QGAS,  lambda s: _q(s, Q_GAS_HI)) if COL_QGAS in df.columns else ("dummy", lambda s: np.nan),
    hz_hi   = (COL_HZ,    lambda s: _q(s, Q_HZ_HI))  if COL_HZ   in df.columns else ("dummy", lambda s: np.nan),
    pint_lo = (COL_PINT,  lambda s: _q(s, Q_PINT_LO))if COL_PINT in df.columns else ("dummy", lambda s: np.nan),
)
print("\n>> Muestra q_reg:")
print(q_reg.head(8))


Merge de percentiles al dataset

Dejamos qtot_lo/hi, gas_hi, hz_hi, pint_lo disponibles por fila.

In [ ]:
# === BLOQUE 5 · MERGE PERCENTILES AL DF ===
df2 = df.merge(q_reg, left_on=[COL_POZO, COL_REG], right_index=True, how="left")
print("\n>> df2 columnas nuevas de sobre operativo añadidas.")
print([c for c in df2.columns if c.endswith(("_lo","_hi"))][:12])


Flags del “sobre” por fila (sin usar slope ahora)

In [ ]:
# === BLOQUE 6 · FLAGS DEL SOBRE OPERATIVO ===
def flag_env_q(row):
    q = row.get(COL_QTOT, np.nan)
    lo, hi = row.get("qtot_lo", np.nan), row.get("qtot_hi", np.nan)
    if pd.isna(q) or pd.isna(lo) or pd.isna(hi):
        return np.nan
    return 1 if (lo <= q <= hi) else 0

def flag_env_gate(row):
    g   = row.get(COL_QGAS,  np.nan)
    gh  = row.get("gas_hi",  np.nan)
    hz  = row.get(COL_HZ,    np.nan)
    hzh = row.get("hz_hi",   np.nan)
    pi  = row.get(COL_PINT,  np.nan)
    pil = row.get("pint_lo", np.nan)

    if pd.isna(g) or pd.isna(gh):
        return 0  # sin gas no penalizamos
    hz_out = (not pd.isna(hz)) and (not pd.isna(hzh)) and (hz > hzh)
    pi_out = (not pd.isna(pi)) and (not pd.isna(pil)) and (pi < pil)
    gas_out = (g > gh) and (hz_out or pi_out)
    return int(gas_out)

df2["env_q"]    = df2.apply(flag_env_q, axis=1)
df2["env_gate"] = df2.apply(flag_env_gate, axis=1)

print("\n>> Conteos env_q / env_gate:")
print(df2["env_q"].value_counts(dropna=False))
print(df2["env_gate"].value_counts(dropna=False))


7) Diagnóstico

In [ ]:
# === BLOQUE 7 · DIAGNÓSTICO RÁPIDO ===
cob = df2[["qtot_lo","qtot_hi","gas_hi","hz_hi","pint_lo"]].notna().mean()
print("\nCobertura (no-NaN) de umbrales efectivos:")
print(cob)

print("\nMuestra filas con env_q NaN (si existieran):")
print(df2[df2["env_q"].isna()][[COL_POZO, COL_FECHA, COL_QTOT, "qtot_lo", "qtot_hi"]].head(8))


8) Señal instantánea de “fuera de sobre”

In [ ]:
# === BLOQUE 8 · ALARMA INSTANTÁNEA (solo sobre) ===
df2["alarma_instantanea"] = np.where(
    (df2["env_q"] == 0) | (df2["env_gate"] == 1),
    1, 0
)

print("\n>> Tabla cruzada: evento_hibrido vs alarma_instantanea (si tu etiqueta existe)")
if "evento_hibrido" in df2.columns:
    print(pd.crosstab(df2["evento_hibrido"], df2["alarma_instantanea"]))
else:
    print("No existe 'evento_hibrido' en df2; solo se muestra la distribución de alarma_instantanea.")
    print(df2["alarma_instantanea"].value_counts())
